In [1]:
# libraries
import pandas as pd
import numpy as np

In [2]:
prod = pd.read_csv('../data/processed/product_level.csv')

feat_cols = ['avg_price', 'avg_rating', 'battery', 'camera', 'performance',
             'design', 'display', 'helpful_votes', 'positive_pct']

print(prod.shape)
print(prod[feat_cols].describe().T[['min', 'max', 'std']].round(2))

(22, 12)
                  min      max     std
avg_price      392.71  1108.27  249.96
avg_rating       3.05     3.15    0.03
battery          2.67     2.75    0.02
camera           2.66     2.78    0.03
performance      2.65     2.77    0.04
design           2.67     2.77    0.03
display          2.65     2.76    0.03
helpful_votes    3.55     3.74    0.05
positive_pct    53.65    57.72    1.12


In [3]:
from sklearn.preprocessing import StandardScaler

feat_cols = ['avg_price', 'avg_rating', 'battery', 'camera', 'performance',
             'design', 'display', 'positive_pct']          # helpful_votes dropped (engagement, not a spec)

scaler_p = StandardScaler()
P = pd.DataFrame(scaler_p.fit_transform(prod[feat_cols]), columns=feat_cols)

weights = {c: 1.0 for c in feat_cols}
weights['avg_price'] = 3.0                                  # price counts about as much as all ratings together
P_w = P * pd.Series(weights)

P_w.round(2).head()

,avg_price,avg_rating,battery,camera,performance,design,display,positive_pct
0,4.63,1.17,1.60,1.27,1.50,1.93,-0.38,1.57
1,4.51,1.98,1.11,2.43,0.07,1.93,1.53,1.87
2,4.57,-1.68,-1.82,-0.28,-1.66,-1.15,-1.15,-1.84
3,4.57,-0.87,0.62,-0.67,0.35,-0.81,0.77,-0.70
4,0.94,-0.46,-0.35,-0.28,-0.80,-0.12,0.00,0.68


In [7]:
print(prod['model'].tolist())

['Iphone 13', 'Iphone 14', 'Iphone 15 Pro', 'Iphone Se', 'Pixel 6', 'Pixel 7A', 'Pixel 8', 'Edge 50', 'Moto G Power', 'Razr 40', 'Oneplus 11R', 'Oneplus 12', 'Oneplus Nord 3', 'Realme 12 Pro', 'Realme Narzo 70', 'Galaxy A55', 'Galaxy Note 20', 'Galaxy S24', 'Galaxy Z Flip', 'Mi 13 Pro', 'Poco X6', 'Redmi Note 13']


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(P_w)
sim_df = pd.DataFrame(sim, index=prod['model'], columns=prod['model'])

def recommend(model_name, n=5):
    lookup = {m.lower(): m for m in prod['model']}
    model_name = lookup[model_name.lower()]
    
    scores = sim_df[model_name].drop(model_name).sort_values(ascending=False).head(n)
    out = prod.set_index('model').loc[scores.index, ['brand', 'avg_price', 'avg_rating']].copy()
    out['similarity'] = scores.round(3)
    return out.reset_index()

recommend('iPhone 14')

,model,brand,avg_price,avg_rating,similarity
0,Iphone 13,Apple,1108.27,3.13,0.897
1,Galaxy Note 20,Samsung,899.75,3.13,0.852
2,Pixel 7A,Google,806.82,3.11,0.767
3,Galaxy Z Flip,Samsung,899.38,3.11,0.734
4,Pixel 8,Google,798.97,3.10,0.651


In [9]:
price = prod.set_index('model')['avg_price']

rec_gap = []
for m in prod['model']:
    r = recommend(m, n=5)
    rec_gap.append((r['avg_price'] - price[m]).abs().mean())

rand_gap = (price.values[:, None] - price.values[None, :])
rand_gap = np.abs(rand_gap).sum() / (len(price) * (len(price) - 1))

print("avg price gap, recommended:", round(np.mean(rec_gap), 1))
print("avg price gap, random pairs:", round(rand_gap, 1))

avg price gap, recommended: 133.5
avg price gap, random pairs: 290.4


In [10]:
import joblib

prod['price_tier'] = pd.cut(prod['avg_price'], bins=[0, 550, 1000, np.inf],
                            labels=['Budget', 'Mid-range', 'Premium'])
print(prod.groupby('price_tier', observed=True)['model'].apply(list))

def recommend(model_name, n=5):
    lookup = {m.lower(): m for m in prod['model']}
    model_name = lookup[model_name.lower()]
    scores = sim_df[model_name].drop(model_name).sort_values(ascending=False).head(n)
    out = prod.set_index('model').loc[scores.index, ['brand', 'price_tier', 'avg_price', 'avg_rating']].copy()
    out['similarity'] = scores.round(3)
    return out.reset_index()

print(recommend('iPhone 14'))

# save for the Streamlit app
prod.to_csv('../data/processed/product_level.csv', index=False)
joblib.dump({'scaler': scaler_p, 'weights': weights, 'feat_cols': feat_cols},
            '../models/recommender.pkl')

price_tier
Budget       [Edge 50, Moto G Power, Razr 40, Realme 12 Pro...
Mid-range    [Pixel 6, Pixel 7A, Pixel 8, Oneplus 11R, Onep...
Premium       [Iphone 13, Iphone 14, Iphone 15 Pro, Iphone Se]
Name: model, dtype: object
            model    brand price_tier  avg_price  avg_rating  similarity
0       Iphone 13    Apple    Premium    1108.27        3.13       0.897
1  Galaxy Note 20  Samsung  Mid-range     899.75        3.13       0.852
2        Pixel 7A   Google  Mid-range     806.82        3.11       0.767
3   Galaxy Z Flip  Samsung  Mid-range     899.38        3.11       0.734
4         Pixel 8   Google  Mid-range     798.97        3.10       0.651


['../models/recommender.pkl']